# Create Science Foundation Ireland (SFI) Awards

**Science Foundation Ireland** (now operating as Taighde Éireann – Research Ireland) is Ireland's main funder of scientific research. This ingest covers SFI grant commitments.

**Source — SFI official open data (Method 1, bulk CSV).** `scripts/local/sfi_to_s3.py` downloads SFI's published Grant Commitments file (`sfi.ie/about-us/governance/open-data/...csv`, also on data.gov.ie) and filters to `Funder Name = Science Foundation Ireland`. A premium per-grant feed.

**Awarding body funder:** Science Foundation Ireland — funder_id **4320320847** (IE; `F4320*` ⇒ Path A, in `openalex.common.funder`).

**Schema choices:**
- `funder_award_id` = the **SFI Proposal ID** (e.g. `00/PI.1/B038`). `display_name` = Proposal Title. `description` NULL (no abstract in the file).
- `amount` = `Current Total Commitment` (**EUR**); **negative/zero de-commitment rows → NULL per §6.7** (never imputed). `currency` = EUR only where an amount survives.
- `lead_investigator` = the Lead Applicant, with **ORCID** carried through (~74%); `affiliation.name` = Research Body, `country` = IE, and **`affiliation.ids` carries the Research Body ROR** (~95%, `type='ror'`, `asserted_by='sfi'`) for exact institution linkage. Institutional/role placeholders ("VP Research (UCD)") are NULL-person (§6.4a, handled in the scraper).
- `funding_type` = fellowship for fellowship/career schemes else grant; `funder_scheme` = Programme (+ Sub-Programme).
- `start_date`/`end_date` real (day precision); future-year cap applied. `co_lead`/`investigators` NULL (file carries lead only).

**S3:** `s3a://openalex-ingest/awards/sfi/sfi_projects.parquet`  ·  **provenance:** `sfi_open_data`


## Step 1: Staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sfi_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/sfi/sfi_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.sfi_raw;


## Step 1.5: Inspect raw + coverage

Expected: ~7,269 SFI grants, 2001-2025. title / lead_family_name / institution / start_date ~100%; ORCID ~74%, ROR ~95%, amount ~98% (EUR).


In [ ]:
%sql
DESCRIBE openalex.awards.sfi_raw;


In [ ]:
%sql
SELECT funder_award_id, lead_given_name, lead_family_name, lead_orcid, institution_name,
       institution_ror, amount, start_date, funding_type, SUBSTRING(title,1,48) AS title
FROM openalex.awards.sfi_raw LIMIT 8;


In [ ]:
%sql
SELECT COUNT(*) AS total,
       COUNT(DISTINCT funder_award_id) AS distinct_ids,
       COUNT(lead_family_name) AS has_lead, COUNT(lead_orcid) AS has_orcid,
       COUNT(institution_ror) AS has_ror, COUNT(amount) AS has_amount,
       COUNT(title) AS has_title, COUNT(start_date) AS has_start
FROM openalex.awards.sfi_raw;


## Step 1.6: Fail-fast — verify SFI funder row exists (Path A)

`4320320847` is `F4320*`, so it MUST be present in `openalex.common.funder`. Must return exactly 1 row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320847;


## Step 2: Transform to award schema

GRANT pattern with named PI (+ ORCID), institution (+ ROR in affiliation.ids), and EUR amount. Negative/zero amounts → NULL (§6.7). Future-year cap (§2.3).


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sfi_awards
USING delta AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320847   -- Science Foundation Ireland
),
base AS (
    SELECT
        n.funder_award_id, n.title, n.funder_scheme, n.funding_type,
        n.lead_given_name, n.lead_family_name, n.lead_orcid,
        n.institution_name, n.institution_ror, n.country,
        TRY_CAST(n.amount AS DOUBLE) AS amt, n.currency,
        TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS sd,
        TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS ed,
        f.funder_id, f.display_name AS f_name, f.ror_id, f.doi AS f_doi
    FROM openalex.awards.sfi_raw n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
)
SELECT
    abs(xxhash64(CONCAT(TRY_CAST(funder_id AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS id,
    title AS display_name,
    CAST(NULL AS STRING) AS description,
    funder_id,
    funder_award_id,
    amt AS amount,
    CASE WHEN amt IS NOT NULL THEN currency ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(funder_id AS STRING)) AS id,
        f_name AS display_name, ror_id AS ror_id, f_doi AS doi
    ) AS funder,
    funding_type,
    funder_scheme,
    'sfi_open_data' AS provenance,
    sd AS start_date,
    ed AS end_date,
    CASE WHEN YEAR(sd) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(sd) END AS start_year,
    CASE WHEN YEAR(sd) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(ed) END AS end_year,
    CASE WHEN lead_given_name IS NOT NULL OR lead_family_name IS NOT NULL OR institution_name IS NOT NULL THEN struct(
        lead_given_name AS given_name,
        lead_family_name AS family_name,
        lead_orcid AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            institution_name AS name,
            country AS country,
            CASE WHEN institution_ror IS NOT NULL
                 THEN ARRAY(struct(institution_ror AS id, 'ror' AS type, 'sfi' AS asserted_by))
                 ELSE CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) END AS ids
        ) AS affiliation
    ) ELSE NULL END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CAST(NULL AS STRING) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(TRY_CAST(funder_id AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM base;


## Step 3: Insert into openalex_awards_raw at priority 169


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sfi_open_data' AND priority = 169;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    169 AS priority  -- Science Foundation Ireland
FROM openalex.awards.sfi_awards;


## Step 6: Verification

§6.7 amount **present** (Current Total Commitment, EUR; ~98% after dropping negative/zero de-commitments). Completeness: display_name / lead_investigator / start_year ~100%.


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.sfi_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.sfi_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total, COUNT(display_name) AS has_title, COUNT(lead_investigator) AS has_lead,
    COUNT(amount) AS has_amount, COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) AS pct_lead,
    collect_set(currency) AS currencies, ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.sfi_awards;


In [ ]:
%sql
-- §6.4a sanity: no single PI/title should dominate; institutional-role names already NULLed in scraper
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.sfi_awards
GROUP BY lead_investigator.given_name, lead_investigator.family_name
ORDER BY n DESC LIMIT 10;


In [ ]:
%sql
SELECT MIN(start_year) AS first_year, MAX(start_year) AS last_year, COUNT(*) AS total,
       COUNT(DISTINCT id) AS distinct_ids,
       SUM(CASE WHEN start_year > YEAR(current_date())+1 THEN 1 ELSE 0 END) AS future_year_rows
FROM openalex.awards.sfi_awards;


In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.sfi_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT id, SUBSTRING(display_name,1,46) AS title, funding_type, amount, start_year,
       lead_investigator.family_name AS family, lead_investigator.orcid AS orcid,
       lead_investigator.affiliation.name AS institution,
       lead_investigator.affiliation.ids[0].id AS ror
FROM openalex.awards.sfi_awards
ORDER BY start_year DESC LIMIT 10;
